In [ ]:
import numpy as np
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
import h5py

import tensorflow as tf
tf.keras.utils.set_random_seed(424)
tf.keras.utils.clear_session(free_memory=True)

import warnings
warnings.filterwarnings("ignore")

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if len(gpus):
    # 设置 GPU 显存占用为按需分配，增长式
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
        	# 异常处理
        	print(e)

In [ ]:
with h5py.File(f"pretrain_data_2064.h5", "r") as rf :
    pretrain_data = rf['pretrain_data'][:]
with h5py.File(f"pretrain_lab_2064.h5", "r") as rf :
    pretrain_lab = rf['pretrain_lab'][:]

In [ ]:
# all subjects ECG data and ann data
with h5py.File(f"sig_mat_2064_1.h5", "r") as rf :
    sig_mat = rf['sig_mat'][:][:,::,1:]
with h5py.File(f"ann_seg_2064_1.h5", "r") as rf :
    ann_seg = rf['ann_seg'][:]

In [4]:
# V3, V2, A3, A2, -V3, -V2, -A3, -A2, VA4, VA5, -VA4, -VA5, L3
# 0,  1,  2,  3,  4,   5,   6,   7,   8,   9,   10,   11,   12

lab_seg = np.zeros((len(ann_seg[:,0]),13))
for line in np.arange(len(ann_seg[:,0])) :
    # ====== V temp =======
    if ann_seg[line,0] <= 3.5 :
        lab_seg[line,0] = 0
        lab_seg[line,1] = 0
    elif ann_seg[line,0] <= 6.5 :
        lab_seg[line,0] = 1
        if ann_seg[line,0] <= 5 :
            lab_seg[line,1] = 0
        else:
            lab_seg[line,1] = 1
    else :
        lab_seg[line,0] = 2
        lab_seg[line,1] = 1
    # ====== A temp =======    
    if ann_seg[line,1] <= 3.5 :
        lab_seg[line,2] = 0
        lab_seg[line,3] = 0
    elif ann_seg[line,1] <= 6.5 :
        lab_seg[line,2] = 1
        if ann_seg[line,1] <= 5 :
            lab_seg[line,3] = 0
        else:
            lab_seg[line,3] = 1
    else :
        lab_seg[line,2] = 2
        lab_seg[line,3] = 1

    # ====== V avg =======
    if ann_seg[line,2] <= 3.5 :
        lab_seg[line,4] = 0
        lab_seg[line,5] = 0
    elif ann_seg[line,2] <= 6.5 :
        lab_seg[line,4] = 1
        if ann_seg[line,2] <= 5 :
            lab_seg[line,5] = 0
        else:
            lab_seg[line,5] = 1
    else :
        lab_seg[line,4] = 2
        lab_seg[line,5] = 1
    # ====== A avg =======    
    if ann_seg[line,3] <= 3.5 :
        lab_seg[line,6] = 0
        lab_seg[line,7] = 0
    elif ann_seg[line,3] <= 6.5 :
        lab_seg[line,6] = 1
        if ann_seg[line,3] <= 5 :
            lab_seg[line,7] = 0
        else:
            lab_seg[line,7] = 1
    else :
        lab_seg[line,6] = 2
        lab_seg[line,7] = 1

    # ======= VA-4 temp ======
    if lab_seg[line,1] == 0 and lab_seg[line,3] == 0 :
        lab_seg[line,8] = 1
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 1 :
        lab_seg[line,8] = 0
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 0 :
        lab_seg[line,8] = 2
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 1 :
        lab_seg[line,8] = 3
    # ======== VA-4 avg ======
    if lab_seg[line,5] == 0 and lab_seg[line,7] == 0 :
        lab_seg[line,10] = 1
    elif lab_seg[line,5] == 0 and lab_seg[line,7] == 1 :
        lab_seg[line,10] = 0
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 0 :
        lab_seg[line,10] = 2
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 1 :
        lab_seg[line,10] = 3 
    
    # ======== VA-5 temp ======
    if lab_seg[line,0] == 1 and lab_seg[line,2] == 1 :
        lab_seg[line,9] = 2
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 0 :
        lab_seg[line,9] = 1
    elif lab_seg[line,1] == 0 and lab_seg[line,3] == 1 :
        lab_seg[line,9] = 0
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 0 :
        lab_seg[line,9] = 3
    elif lab_seg[line,1] == 1 and lab_seg[line,3] == 1 :
        lab_seg[line,9] = 4
    # ======== VA-4 avg ======
    if lab_seg[line,4] == 1 and lab_seg[line,6] == 1 :
        lab_seg[line,11] = 2
    elif lab_seg[line,5] == 0 and lab_seg[line,7] == 0 :
        lab_seg[line,11] = 1
    elif lab_seg[line,5] == 0 and lab_seg[line,7] == 1 :
        lab_seg[line,11] = 0
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 0 :
        lab_seg[line,11] = 3
    elif lab_seg[line,5] == 1 and lab_seg[line,7] == 1 :
        lab_seg[line,11] = 4 

    # ======== L3 =========
    if ann_seg[line,4] ==0 or ann_seg[line,4] == 1 :
        lab_seg[line,12] = 0
    elif ann_seg[line,4] ==6 or ann_seg[line,4] == 7 :
        lab_seg[line,12] = 2
    else:
        lab_seg[line,12] = 1

In [5]:
class FFTLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # 对每个 channel 进行 FFT 变换
        fft_result1 = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part1 = tf.math.real(fft_result1)*amp_norm
        imag_part1 = tf.math.imag(fft_result1)*amp_norm

        # 对每个 channel 进行 FFT 变换
        fft_result2 = tf.signal.fft(tf.cast(inputs[:,:,1], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part2 = tf.math.real(fft_result2)*amp_norm
        imag_part2 = tf.math.imag(fft_result2)*amp_norm

        # 对每个 channel 进行 FFT 变换
        fft_result3 = tf.signal.fft(tf.cast(inputs[:,:,2], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        amp_norm = np.ones((1,1280))/640
        amp_norm[:,0] = 0
        real_part3 = tf.math.real(fft_result3)*amp_norm
        imag_part3 = tf.math.imag(fft_result3)*amp_norm
        
        return tf.concat([real_part1[:,:,tf.newaxis], imag_part1[:,:,tf.newaxis],
                          real_part2[:,:,tf.newaxis], imag_part2[:,:,tf.newaxis],
                          real_part3[:,:,tf.newaxis], imag_part3[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])

In [6]:
class FFTLayer1D(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FFTLayer1D, self).__init__(**kwargs)

    def call(self, inputs):
        # 对每个 channel 进行 FFT 变换
        fft_result1 = tf.signal.fft(tf.cast(inputs[:,:,0], tf.complex64))
        # 将实部和虚部沿着最后一个维度拼接
        real_part1 = tf.math.real(fft_result1)
        imag_part1 = tf.math.imag(fft_result1)
        
        return tf.concat([real_part1[:,:,tf.newaxis], imag_part1[:,:,tf.newaxis]], axis=-1)

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, 2 * channel)
        return (input_shape[0], input_shape[1], 2 * input_shape[2])

In [7]:
class IFFTLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(IFFTLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # 将输入分为实部和虚部
        real_part = inputs[..., 0]
        imag_part = inputs[..., 1]
        # 组合成复数
        complex_input = tf.complex(real_part, imag_part)
        # 对每个 channel 进行 IFFT 变换
        ifft_result = tf.signal.ifft(complex_input)
        return tf.math.real(ifft_result)[:,:,tf.newaxis]

    def compute_output_shape(self, input_shape):
        # 输出维度为 (batch, length, channel)
        return (input_shape[0], input_shape[1], input_shape[2] // 2)

In [8]:
class CrossTFAttention(tf.keras.layers.Layer):
    def __init__(self, attnum, num_head, key_dim, output_shape, **kwargs):
        super(CrossTFAttention, self).__init__(name=f'crftatt{attnum}_block', **kwargs)
        self.frqcov = tf.keras.layers.Conv1D(
            filters=output_shape,
            kernel_size=3,
            padding='same',
            name=f'crftatt{attnum}_frqcov')

        self.ifft = IFFTLayer()
        self.outshape = output_shape
        # self.mhatt = tf.keras.layers.MultiHeadAttention(num_heads=num_head,key_dim=key_dim,output_shape=output_shape,dropout=0.1,name=f'crftatt{attnum}_mhatt')
        self.add = tf.keras.layers.Add(name=f'crftatt{attnum}_mhatt')
        # self.attadd = tf.keras.layers.Add(name=f'crftatt{attnum}_add')

    def call(self, frq, sig) :
        frq_components = tf.split(frq, num_or_size_splits=int(self.outshape/2), axis=-1)
        frq_outputs = []
        for i in range(int(self.outshape/2)) :
            frq_sig = self.ifft(frq_components[i])
            frq_outputs.append(frq_sig)
        ftime_sig = tf.concat(frq_outputs, axis=2)
        frq_cp_i = self.frqcov(ftime_sig)
        # timfreq_att = self.mhatt(query=frq_cp_i,value=sig,key=sig)
        # resadd_att = self.attadd([sig,timfreq_att,frq_cp_i])
        resadd_att = self.add([sig,frq_cp_i])
        
        return resadd_att 

In [9]:
class MultiConv1DLayer(tf.keras.layers.Layer):
    def __init__(self, cov_size, splits_size, trainable=True, filters=1, kernel_size=1, activation='gelu', **kwargs):
        """
        自定义多 Conv1D 层
        
        参数:
        cov_size -- 表示要使用的 Conv1D 层的数量
        filters -- 每个 Conv1D 层的过滤器数量，默认为32
        kernel_size -- 每个 Conv1D 层的核大小，默认为3
        activation -- 每个 Conv1D 层的激活函数，默认为'relu'
        """
        super(MultiConv1DLayer, self).__init__(**kwargs)
        self.cov_size = cov_size
        self.splits_size = splits_size
        self.filters = filters
        self.kernel_size = kernel_size
        self.activation = activation
        self.conv_layers = []
        self.trainable = trainable

    def build(self, input_shape):
        # 创建 cov_size 个 Conv1D 层
        for _ in range(self.cov_size):
            self.conv_layers.append(tf.keras.layers.Conv1D(
                filters=self.filters,
                kernel_size=self.kernel_size,
                padding="same",
                kernel_initializer=tf.keras.initializers.HeNormal(),
                trainable = self.trainable,
                activation=self.activation
            ))
        super(MultiConv1DLayer, self).build(input_shape)

    def call(self, inputs):               
        # 每个 Conv1D 层处理输入的一个分量
        outputs = []
        if self.splits_size == 3 :
            # 分割输入张量，使得每个 Conv1D 层可以处理输入的不同分量
            input_components = tf.split(inputs, num_or_size_splits=3, axis=2) 
            for i in range(self.cov_size - 2):
                output_i = self.conv_layers[i](input_components[int(i/2)])# + input_components[int(i/2)]
                outputs.append(output_i)
            outputs.append(self.conv_layers[-2](inputs))
            outputs.append(self.conv_layers[-1](inputs))
        elif self.splits_size == 6 :
            # 分割输入张量，使得每个 Conv1D 层可以处理输入的不同分量
            input_components = tf.split(inputs, num_or_size_splits=6, axis=2) 
            for i in range(self.cov_size - 2):
                output_i = self.conv_layers[i](input_components[i])# + input_components[i]
                outputs.append(output_i)
            outputs.append(self.conv_layers[-2](inputs[:,:,0:6:2]))
            outputs.append(self.conv_layers[-1](inputs[:,:,1:6:2]))
        else :
            # 分割输入张量，使得每个 Conv1D 层可以处理输入的不同分量
            input_components = tf.split(inputs, num_or_size_splits=self.splits_size, axis=2) 
            for i in range(self.cov_size - 1):
                output_i = self.conv_layers[i](input_components[int(i/2)])# + input_components[int(i/2)]
                outputs.append(output_i)
            outputs.append(self.conv_layers[-1](inputs))
        
        # 拼接所有 Conv1D 层的输出
        return tf.concat(outputs, axis=2)
    
    def get_config(self):
        config = super(MultiConv1DLayer, self).get_config()
        config.update({
            'cov_size': self.cov_size,
            'splits_size': self.splits_size,
            'filters': self.filters,
            'kernel_size': self.kernel_size,
            'activation': self.activation
        })
        return config

In [10]:
class ResBlock1D(tf.keras.layers.Layer):
    def __init__(self, covfilter, resnum, splits_size, covks=3, covstrides=1, rb_trainable=True, **kwargs):
        super(ResBlock1D, self).__init__(name=f'rb{resnum}_block', **kwargs)

        self.rescov = MultiConv1DLayer(
            cov_size=covfilter,
            splits_size = splits_size,
            trainable=rb_trainable,
            name=f'rb{resnum}_rescov'
        )

        self.res1_bn1 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res1_bn1')
        self.res1_act1 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res1_act1')
        self.res1_cov1 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            padding='same',
            dilation_rate=1,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res1_cov1'
        )
        self.res1_bn2 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res1_bn2')
        self.res1_act2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res1_act2')        
        self.res1_cov2 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            padding='same',
            dilation_rate=2,
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res1_cov2'
        )
        self.add1 = tf.keras.layers.Add(name=f'rb{resnum}_res1_add')

        self.res2_bn1 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res2_bn1')
        self.res2_act1 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res2_act1')
        self.res2_cov1 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            dilation_rate=3,
            padding='same',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res2_cov1'
        )
        self.res2_bn2 = tf.keras.layers.LayerNormalization(name=f'rb{resnum}_res2_bn2')
        self.res2_act2 = tf.keras.layers.Activation('gelu',name=f'rb{resnum}_res2_act2')        
        self.res2_cov2 = tf.keras.layers.Conv1D(
            filters=covfilter,
            kernel_size=covks,
            dilation_rate=4,
            padding='same',
            kernel_initializer=tf.keras.initializers.HeNormal(),
            trainable=rb_trainable,
            name=f'rb{resnum}_res2_cov2'
        )
        self.add2 = tf.keras.layers.Add(name=f'rb{resnum}_res2_add')


    def call(self, x):
        rescov = self.rescov(x)
        
        res1_bn1 = self.res1_bn1(rescov)
        res1_act1 = self.res1_act1(res1_bn1)
        res1_cov1 = self.res1_cov1(res1_act1)
        res1_bn2 = self.res1_bn2(res1_cov1)
        res1_act2 = self.res1_act2(res1_bn2)
        res1_cov2 = self.res1_cov2(res1_act2)        
        res1_add = self.add1([rescov, res1_cov2])

        res2_bn1 = self.res2_bn1(res1_add)
        res2_act1 = self.res2_act1(res2_bn1)
        res2_cov1 = self.res2_cov1(res2_act1)
        res2_bn2 = self.res2_bn2(res2_cov1)
        res2_act2 = self.res2_act2(res2_bn2)
        res2_cov2 = self.res2_cov2(res2_act2)        
        res2_add = self.add2([res1_add, res2_cov2])

        return res2_add

In [11]:
def BMAutoEncoder(fs: int=64) -> tf.keras.Model:
    # =========================================== signal layers resUnet ===============================================
    # =========================================== FFT layers resUnet ==================================================
    ipl = tf.keras.Input((20*fs,3))
    FFT_layer = FFTLayer()(ipl)
    # =========================================== signal Encoder ======================================================
    res1 = ResBlock1D(8,1,3)(ipl)
    res1_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res1_mp")(res1)
    res2 = ResBlock1D(16,2,8,covks=3)(res1_mp)
    res2_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res2_mp")(res2)
    res3 = ResBlock1D(32,3,16,covks=3)(res2_mp)
    res3_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res3_mp")(res3)
    res4 = ResBlock1D(64,4,32,covks=3)(res3_mp)
    res4_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res4_mp")(res4)
    res_enc = ResBlock1D(128,5,64,covks=3)(res4_mp)

    # ============================================ FFT Encoder =======================================================
    res21 = ResBlock1D(8,21,6)(FFT_layer)
    res21_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res21_mp")(res21)
    res22 = ResBlock1D(16,22,8,covks=3)(res21_mp)
    res22_mp = tf.keras.layers.MaxPool1D(pool_size=4,name="res22_mp")(res22)
    res23 = ResBlock1D(32,23,16,covks=3)(res22_mp)
    res23_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res23_mp")(res23)
    res24 = ResBlock1D(64,24,32,covks=3)(res23_mp)
    res24_mp = tf.keras.layers.MaxPool1D(pool_size=2,name="res24_mp")(res24)
    res_encfft = ResBlock1D(128,25,64,covks=3)(res24_mp)

    # ============================================= Dense ==============================================================
    ctfa_scale1 = CrossTFAttention(attnum=1, num_head=4, key_dim=16, output_shape=8)(frq=res21,sig=res1)
    ctfa_scale2 = CrossTFAttention(attnum=2, num_head=4, key_dim=16, output_shape=16)(frq=res22,sig=res2)
    ctfa_scale3 = CrossTFAttention(attnum=3, num_head=4, key_dim=16, output_shape=32)(frq=res23,sig=res3)
    ctfa_scale4 = CrossTFAttention(attnum=4, num_head=4, key_dim=16, output_shape=64)(frq=res24,sig=res4)
    # ctfa_scale_enc = CrossTFAttention(attnum=4, num_head=4, key_dim=16, output_shape=64)(frq=res_encfft,sig=res_enc)
    ctfa_scale_enc = CrossTFAttention(attnum=5, num_head=4, key_dim=16, output_shape=128)(frq=res_encfft,sig=res_enc)
    
    up5 = tf.keras.layers.UpSampling1D(size=64,name="up5")(ctfa_scale_enc)
    up4 = tf.keras.layers.UpSampling1D(size=32,name="up4")(ctfa_scale4)
    # up4 = tf.keras.layers.UpSampling1D(size=32,name="up4")(ctfa_scale_enc)
    up3 = tf.keras.layers.UpSampling1D(size=16,name="up3")(ctfa_scale3)
    up2 = tf.keras.layers.UpSampling1D(size=4,name="up2")(ctfa_scale2)

    up15 = tf.keras.layers.UpSampling1D(size=64,name="up15")(res_encfft)
    up14 = tf.keras.layers.UpSampling1D(size=32,name="up14")(res24)
    # up14 = tf.keras.layers.UpSampling1D(size=32,name="up14")(res_encfft)
    up13 = tf.keras.layers.UpSampling1D(size=16,name="up13")(res23)
    up12 = tf.keras.layers.UpSampling1D(size=4,name="up12")(res22)

    tenc_cc = tf.keras.layers.concatenate([ctfa_scale1,up2,up3,up4,up5],name="tenc_cc")
    fenc_cc = tf.keras.layers.concatenate([res21,up12,up13,up14,up15],name="fenc_cc")

    sig_deco = tf.keras.layers.MultiHeadAttention(num_heads=4,key_dim=16,kernel_initializer="he_normal",dropout=0.1,name="sig_deco")(tenc_cc,tenc_cc)
    sig_dec_res = tf.keras.layers.Add(name="sig_dec_res")([tenc_cc,sig_deco])
    sig_dec_ln = tf.keras.layers.LayerNormalization(name='sig_dec_ln')(sig_dec_res)
    sig_dec_ffn = tf.keras.layers.Dense(units=248,activation='gelu',kernel_initializer='he_normal',name='sig_dec_ffn')(sig_dec_ln)
    sig_dec_ffnres = tf.keras.layers.Add(name="sig_dec_ffnres")([sig_dec_ffn,sig_dec_ln])
    sig_dec_ffnln = tf.keras.layers.LayerNormalization(name='sig_dec_ffnln')(sig_dec_ffnres)
    
    sig_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='sig_deco_gap')(sig_dec_ffnln)
    sig_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='sig_deco_gmp')(sig_dec_ffnln)
    sig_deco_cc = tf.keras.layers.concatenate([sig_deco_gap,sig_deco_gmp],name="sig_deco_cc")
    sig_dec_f = tf.keras.layers.Dense(units=16,name="sig_dec_f")(sig_deco_cc)
    sig_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="sig_dec_gelu")(sig_dec_f)

    frq_deco = tf.keras.layers.MultiHeadAttention(num_heads=4,key_dim=16,kernel_initializer="he_normal",dropout=0.1,name="frq_deco")(fenc_cc,fenc_cc)
    frq_dec_res = tf.keras.layers.Add(name="frq_dec_res")([fenc_cc,frq_deco])
    frq_dec_ln = tf.keras.layers.LayerNormalization(name='frq_dec_ln')(frq_dec_res)
    frq_dec_ffn = tf.keras.layers.Dense(units=248,activation='gelu',kernel_initializer='he_normal',name='frq_dec_ffn')(frq_dec_ln)
    frq_dec_ffnres = tf.keras.layers.Add(name="frq_dec_ffnres")([frq_dec_ffn,frq_dec_ln])
    frq_dec_ffnln = tf.keras.layers.LayerNormalization(name='frq_dec_ffnln')(frq_dec_ffnres)
    
    frq_deco_gap = tf.keras.layers.GlobalAvgPool1D(name='frq_deco_gap')(frq_dec_ffnln)
    frq_deco_gmp = tf.keras.layers.GlobalMaxPool1D(name='frq_deco_gmp')(frq_dec_ffnln)
    frq_deco_cc = tf.keras.layers.concatenate([frq_deco_gap,frq_deco_gmp],name="frq_deco_cc")
    frq_dec_f = tf.keras.layers.Dense(units=16,name="frq_dec_f")(frq_deco_cc)
    frq_dec_gelu = tf.keras.layers.Activation(activation='tanh',name="frq_dec_gelu")(frq_dec_f)

    ft_cc = tf.keras.layers.concatenate([sig_deco_cc,frq_deco_cc],name="ft_cc")
    ft_fc = tf.keras.layers.Dense(units=16,activation='tanh',name="ft_fc")(ft_cc)
    # ft_add = tf.keras.layers.Add(name="ft_add")([ft_fc,sig_dec_gelu,frq_dec_gelu])

    sig_cls1 = tf.keras.layers.Dense(units=8,activation='softmax',name='sig_cls1')(sig_dec_gelu)
    sig_cls2 = tf.keras.layers.Dense(units=8,activation='softmax',name='sig_cls2')(sig_dec_gelu)
    sig_cls3 = tf.keras.layers.Dense(units=8,activation='softmax',name='sig_cls3')(sig_dec_gelu)

    frq_cls1 = tf.keras.layers.Dense(units=8,activation='softmax',name='frq_cls1')(frq_dec_gelu)
    frq_cls2 = tf.keras.layers.Dense(units=8,activation='softmax',name='frq_cls2')(frq_dec_gelu)
    frq_cls3 = tf.keras.layers.Dense(units=8,activation='softmax',name='frq_cls3')(frq_dec_gelu)

    mrg_cls1 = tf.keras.layers.Dense(units=8,activation='softmax',name='mrg_cls1')(ft_fc)
    mrg_cls2 = tf.keras.layers.Dense(units=8,activation='softmax',name='mrg_cls2')(ft_fc)
    mrg_cls3 = tf.keras.layers.Dense(units=8,activation='softmax',name='mrg_cls3')(ft_fc)

    runet_basemd = tf.keras.Model(inputs=ipl,outputs=[
                                                        sig_cls1,sig_cls2,sig_cls3,
                                                         frq_cls1,frq_cls2,frq_cls3,
                                                         mrg_cls1,mrg_cls2,mrg_cls3
                                                    ],name='runet_basemd')
    
    return runet_basemd

In [12]:
tstmd = BMAutoEncoder()
print(tstmd.summary())

2025-12-29 14:43:37.286815: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38104 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:1b:00.0, compute capability: 8.0


Model: "runet_basemd"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1280, 3)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_layer           │ (None, 1280, 6)   │          0 │ input_layer[0][0] │
│ (FFTLayer)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb21_block          │ (None, 1280, 8)   │        884 │ fft_layer[0][0]   │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb1_block           │ (None, 1280, 8)   │        884 │ input_layer[0][0] │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res21_mp            │ (None, 320, 8)    │          0 │ rb21_block[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_mp             │ (None, 320, 8)    │          0 │ rb1_block[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb22_block          │ (None, 320, 16)   │      3,303 │ res21_mp[0][0]    │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb2_block           │ (None, 320, 16)   │      3,303 │ res1_mp[0][0]     │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res22_mp            │ (None, 80, 16)    │          0 │ rb22_block[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_mp             │ (None, 80, 16)    │          0 │ rb2_block[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb23_block          │ (None, 80, 32)    │     12,751 │ res22_mp[0][0]    │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb3_block           │ (None, 80, 32)    │     12,751 │ res2_mp[0][0]     │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res23_mp            │ (None, 40, 32)    │          0 │ rb23_block[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res3_mp             │ (None, 40, 32)    │          0 │ rb3_block[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb24_block          │ (None, 40, 64)    │     50,079 │ res23_mp[0][0]    │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rb4_block           │ (None, 40, 64)    │     50,079 │ res3_mp[0][0]     │
│ (ResBlock1D)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res24_mp            │ (None, 20, 64)    │          0 │ rb24_block[0][0]

 Total params: 850,304 (3.24 MB)

 Trainable params: 850,304 (3.24 MB)

 Non-trainable params: 0 (0.00 B)

None


In [13]:
def EmoclsBM(bm: tf.keras.Model, cls=3) -> tf.keras.Model:
    
    ft_sig = bm.get_layer("sig_dec_gelu").output
    sig_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='sig_cls')(ft_sig)
    
    ft_frq = bm.get_layer("frq_dec_gelu").output
    frq_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='frq_cls')(ft_frq)

    ft_merg = bm.get_layer("ft_fc").output
    merg_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='merg_cls')(ft_merg)

    ft_cc = bm.get_layer("ft_cc").output
    emo_router = tf.keras.layers.Dense(units=3,activation='sigmoid',name='emo_router')(ft_cc)
    emo_vot_sig = tf.keras.layers.Multiply(name='emo_vot_sig')([emo_router[:,0],sig_cls])
    emo_vot_frq = tf.keras.layers.Multiply(name='emo_vot_frq')([emo_router[:,1],frq_cls])
    emo_vot_mrg = tf.keras.layers.Multiply(name='emo_vot_mrg')([emo_router[:,2],merg_cls])
    emo_vot_add = tf.keras.layers.Add(name='emo_vot_add')([emo_vot_sig,emo_vot_frq,emo_vot_mrg])
    # emo_cls = tf.keras.layers.Dense(units=cls,activation='softmax',name='emo_cls')(ft_add)
    emo_cls = tf.keras.layers.Activation(activation='softmax',name='emo_cls')(emo_vot_add)

    
    emocls = tf.keras.Model(inputs=bm.input,outputs=[emo_cls,sig_cls,frq_cls,merg_cls])

    return emocls

In [14]:
class WarmUpCosineDecayRestarts(tf.keras.optimizers.schedules.LearningRateSchedule):
    """学习率预热加上余弦退火重启"""
    def __init__(
        self,
        initial_learning_rate,
        first_decay_steps,
        t_mul=2.0,
        m_mul=1.0,
        alpha=0.0,
        warm_step=1000,
        min_lr=0.0,
        max_lr=1e-3,
        name=None,
    ):
        super().__init__()
        self.initial_learning_rate = initial_learning_rate
        self.first_decay_steps = first_decay_steps
        self.t_mul = t_mul
        self.m_mul = m_mul
        self.alpha = alpha
        self.cosine_decay_restarts = tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=initial_learning_rate,
            first_decay_steps=first_decay_steps,
            t_mul=t_mul,
            m_mul=m_mul,
            alpha=alpha
        )
        self.warm_step = warm_step
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.name = name

    def __call__(self, step):
        with tf.name_scope(self.name or "WarmUpCosineDecayRestarts"):
            # 使用 tf.cond 处理条件逻辑
            def warm_up_learning_rate():
                return tf.cast(
                    self.min_lr + (self.max_lr - self.min_lr) * step / self.warm_step,
                    tf.float32,
                )

            def cosine_decay_learning_rate():
                return self.cosine_decay_restarts(step - self.warm_step)

            return tf.cond(
                step < self.warm_step,
                warm_up_learning_rate,
                cosine_decay_learning_rate
            )

    def get_config(self):
        return {
            "initial_learning_rate": self.initial_learning_rate,
            "first_decay_steps": self.first_decay_steps,
            "t_mul": self.t_mul,
            "m_mul": self.m_mul,
            "alpha": self.alpha,
            "warm_step": self.warm_step,
            "min_lr": self.min_lr,
            "max_lr": self.max_lr,
            "name": self.name,
        }



# Pre-training

In [ ]:
tf.keras.utils.set_random_seed(424)
epc = 100
bs = 64
lr = 1e-4
lr_fn = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate = lr,
    first_decay_steps = 103*5,
    alpha=0.0001
)

# ============================= Loss Matrix for Encoder part ====================================
tf.keras.utils.clear_session(free_memory=True)

X_train,X_test,Y_train,Y_test = train_test_split(pretrain_data,pretrain_lab,test_size=0.1)

Ytrain_onehot1 = tf.keras.utils.to_categorical(Y_train[:,0], num_classes=8)
Ytrain_onehot2 = tf.keras.utils.to_categorical(Y_train[:,1], num_classes=8)
Ytrain_onehot3 = tf.keras.utils.to_categorical(Y_train[:,2], num_classes=8)

Ytest_onehot1 = tf.keras.utils.to_categorical(Y_test[:,0], num_classes=8)
Ytest_onehot2 = tf.keras.utils.to_categorical(Y_test[:,1], num_classes=8)
Ytest_onehot3 = tf.keras.utils.to_categorical(Y_test[:,2], num_classes=8)

ptp_emocls = BMAutoEncoder()
# ptp_emocls.load_weights("case_ptp_md_pretrain.valbest.keras")

opt_ptp = tf.keras.optimizers.AdamW(learning_rate=lr_fn, clipnorm=1.)
ptp_emocls.compile(loss=['categorical_crossentropy','categorical_crossentropy','categorical_crossentropy',
                        'categorical_crossentropy','categorical_crossentropy','categorical_crossentropy',
                        'categorical_crossentropy','categorical_crossentropy','categorical_crossentropy'
                        ],
                   loss_weights=[
                                   1,    1,    1,
                                    1.5,  1.5,  1.65,
                                    1,    1.15,    1
                               ],
                   optimizer=opt_ptp,
                   metrics=["categorical_accuracy","categorical_accuracy","categorical_accuracy",
                           "categorical_accuracy","categorical_accuracy","categorical_accuracy",
                           "categorical_accuracy","categorical_accuracy","categorical_accuracy"
                           ])
ckpt_ptp_fp = f"case_ptp_md_pretrain.valbest.keras"
ckpt_ptp = tf.keras.callbacks.ModelCheckpoint(ckpt_ptp_fp, monitor='val_mrg_cls1_categorical_accuracy', verbose=2, save_best_only=True, mode='auto')
ptp_his = ptp_emocls.fit(x=X_train[:,:,:], y=[Ytrain_onehot1,Ytrain_onehot2,Ytrain_onehot3,
                                             Ytrain_onehot1,Ytrain_onehot2,Ytrain_onehot3,
                                             Ytrain_onehot1,Ytrain_onehot2,Ytrain_onehot3], 
                         batch_size=bs, epochs=epc, verbose=2, 
                         validation_data=(X_test[:,:,:],[Ytest_onehot1,Ytest_onehot2,Ytest_onehot3,
                                                        Ytest_onehot1,Ytest_onehot2,Ytest_onehot3,
                                                        Ytest_onehot1,Ytest_onehot2,Ytest_onehot3]), 
                         callbacks=[ckpt_ptp])


In [37]:
tf.keras.utils.clear_session(free_memory=True)

# A-3 as an example

In [ ]:
tf.keras.utils.set_random_seed(424)
epc = 50
bs = 32
# lr = 5e-5
lr = 1e-4
allsub_predv = np.array([])
ptp_accv,ptp_f1v = np.zeros((30)),np.zeros((30))
# ============================= Loss Matrix for Encoder part ====================================
runet_his_loss = np.zeros((30,epc))
runet_his_valloss = np.zeros((30,epc))
lr_fn = WarmUpCosineDecayRestarts(
    initial_learning_rate = lr,
    first_decay_steps = 103*3,
    alpha=0.001,
    warm_step=206*10,
    min_lr=1e-8,
    max_lr=1e-4    
)
ptp_bsmdprtr = BMAutoEncoder()
# ============================================================================= LOMO-CV =====================================================================================
for sub in np.arange(start=0,stop=30,step=1) :
    print(f'Testing for Subject {sub+1} ...',flush=True)
    # ========================== label check ===================================
    train_lab = lab_seg[ann_seg[:,-1]!=sub,4]
    test_lab = lab_seg[ann_seg[:,-1]==sub,4]
    
    train_onehot_lab = tf.keras.utils.to_categorical(train_lab, num_classes=3)
    test_onehot_lab = tf.keras.utils.to_categorical(test_lab, num_classes=3)
    
    spw = class_weight.compute_sample_weight(class_weight='balanced', y=train_lab)
    # spw2 = class_weight.compute_sample_weight(class_weight='balanced', y=train_onehot_lab)
    # =========================== data check ====================================
    train_sig = sig_mat[ann_seg[:,-1]!=sub,::,:]
    test_sig = sig_mat[ann_seg[:,-1]==sub,::,:]
    # =========================== Training Divide ===============================
    # X_train,X_test,Y_train,Y_test = train_test_split(train_sig,train_lab,test_size=0.2)
    # Ytrain_onehot_lab = tf.keras.utils.to_categorical(Y_train, num_classes=3)
    # Ytest_onehot_lab = tf.keras.utils.to_categorical(Y_test, num_classes=3)
    # spw = class_weight.compute_sample_weight(class_weight='balanced', y=Y_train)
    # =========================== model check ====================================

    ptp_bsmdprtr.load_weights("case_ptp_md_pretrain.valbest.keras")
    ptp_emocls = EmoclsBM(bm=ptp_bsmdprtr, cls=3)
    
    opt_ptp = tf.keras.optimizers.AdamW(learning_rate=lr_fn, weight_decay=0.004, clipnorm=1.) # weight_decay = 0.004
    ptp_emocls.compile(loss=['categorical_crossentropy','categorical_crossentropy','categorical_crossentropy','categorical_crossentropy'],
                       optimizer=opt_ptp,
                       metrics=[["categorical_accuracy",tf.keras.metrics.F1Score(average="weighted"),tf.keras.metrics.F1Score(average="macro")],
                                ["categorical_accuracy"],
                                ["categorical_accuracy"],
                                ["categorical_accuracy"]
                               ])
    ckpt_ptp_fp = f"case_md_mout_res/avg_v3_a3/case_ptp_cls_loso_train_v3_s{sub}.valbest.keras"
    ckpt_ptp = tf.keras.callbacks.ModelCheckpoint(ckpt_ptp_fp, monitor='val_emo_cls_categorical_accuracy', verbose=2, save_best_only=True, mode='auto')
    ptp_his = ptp_emocls.fit(x=train_sig[:,:,:], y=[train_onehot_lab,train_onehot_lab,train_onehot_lab,train_onehot_lab], 
                             batch_size=bs, epochs=epc, verbose=2, sample_weight=spw, 
                             validation_data=(test_sig[:,:,:],[test_onehot_lab,test_onehot_lab,test_onehot_lab,test_onehot_lab]),
                             callbacks=[ckpt_ptp]) 
    tf.keras.utils.clear_session(free_memory=True)